In [ ]:
import pandas as pd
import cwms
import os
import logging


TS_NAME = "KEYS.Elev.Inst.1Hour.0.Ccp-Rev"
OFFICE = os.getenv("OFFICE", "SWT")
API_KEY_DEV = os.getenv("CDA_API_KEY_DEV")
API_ROOT_DEV = os.getenv("CDA_API_ROOT_DEV")

logging.basicConfig(
    level=logging.DEBUG, format="%(asctime)s - %(levelname)s - %(message)s"
)


if not API_ROOT_DEV:
    logging.error(
        "CDA_API_KEY environment variable is not set. Please set it to your API key."
    )

if not API_KEY_DEV:
    logging.error(
        "CDA_API_ROOT environment variable is not set. Please set it to your API root URL."
    )

logging.debug("Starting Script!")


logging.info(f"Getting timeseries data for {TS_NAME} from office {OFFICE}")

cwms.init_session()
ts = cwms.get_timeseries(ts_id=TS_NAME, office_id=OFFICE)
shifted_df = ts.df.copy()

if isinstance(shifted_df.index, pd.DatetimeIndex):
    shifted_df.index = shifted_df.index + pd.Timedelta(hours=10)
else:
    shifted_df["date-time"] = pd.to_datetime(shifted_df["date-time"]) + pd.Timedelta(
        hours=10
    )

ts_json = cwms.timeseries_df_to_json(
    data=shifted_df,
    office_id=OFFICE,
    ts_id=TS_NAME,
    units=ts.json.get("units"),
)
cwms.init_session(api_root=API_ROOT_DEV, api_key=API_KEY_DEV)
cwms.store_timeseries(data=ts_json)


logging.debug("Script Done")
